# Transcribe on a free Kaggle GPU

1. Settings → Accelerator → **GPU** (T4 or P100).
2. Settings → Internet → **On**.
3. Add the video as a dataset input.
4. Set `VIDEO` in the last cell if more than one media file is attached.
5. Run all cells. The notebook converts the video to MP3, then runs faster-whisper `large-v3`. The transcript is saved in `/kaggle/working`.


In [ ]:
import shutil
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"])
if shutil.which("ffmpeg") is None:
    subprocess.check_call(["apt-get", "update"])
    subprocess.check_call(["apt-get", "install", "-y", "ffmpeg"])


In [ ]:
import sys
from pathlib import Path

Path('audio_extract.py').write_text('"""Convert a video or audio file to a temporary 16 kHz mono MP3."""\n\nimport os\nimport subprocess\nimport tempfile\nfrom pathlib import Path\n\n\ndef extract_mp3(source: Path, offset_seconds: int = 0) -> Path:\n    """Extract speech audio to a temporary MP3. Caller deletes the file."""\n    if not source.exists():\n        raise FileNotFoundError(source)\n\n    fd, raw_path = tempfile.mkstemp(suffix=".mp3")\n    os.close(fd)\n    destination = Path(raw_path)\n    command = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error"]\n    if offset_seconds > 0:\n        command.extend(["-ss", str(offset_seconds)])\n    command.extend(\n        [\n            "-i",\n            str(source),\n            "-vn",\n            "-ac",\n            "1",\n            "-ar",\n            "16000",\n            "-c:a",\n            "libmp3lame",\n            "-b:a",\n            "64k",\n            str(destination),\n        ]\n    )\n    result = subprocess.run(command, capture_output=True, text=True)\n    if result.returncode != 0:\n        destination.unlink(missing_ok=True)\n        raise RuntimeError(f"ffmpeg failed:\\n{result.stderr}")\n    return destination\n', encoding='utf-8')
Path('gpu_transcribe.py').write_text('"""Transcribe a video on a free CUDA GPU (Colab T4, Kaggle T4/P100).\n\nConverts the file to a 16 kHz mono MP3 first, then runs faster-whisper large-v3.\n"""\n\nimport argparse\nimport glob\nimport os\nimport shutil\nimport site\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nfrom audio_extract import extract_mp3\n\n# Same filler phrases the local transcriber drops during silence.\nHALLUCINATED_PHRASES = {\n    "thank you",\n    "thanks for watching",\n    "thank you for watching",\n    "thank you very much",\n    "thanks",\n    "bye",\n    "bye bye",\n    "you",\n    ".",\n    ",",\n    "...",\n}\n\nVIDEO_SUFFIXES = {".mkv", ".mp4", ".mov", ".avi", ".webm", ".m4a", ".mp3", ".wav", ".flac"}\n\n\ndef require_gpu() -> str:\n    if shutil.which("nvidia-smi") is None:\n        raise SystemExit(\n            "No GPU detected. Colab: Runtime → Change runtime type → T4 GPU. "\n            "Kaggle: Settings → Accelerator → GPU, and enable Internet."\n        )\n    name = subprocess.check_output(\n        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],\n        text=True,\n    ).strip()\n    if not name:\n        raise SystemExit("nvidia-smi did not report a GPU.")\n    return name.splitlines()[0].strip()\n\n\ndef ensure_cuda_libs() -> None:\n    """Make libcublas loadable. Colab and Kaggle usually already have it."""\n    import ctypes\n\n    try:\n        ctypes.CDLL("libcublas.so.12")\n        return\n    except OSError:\n        pass\n\n    subprocess.check_call(\n        [\n            sys.executable,\n            "-m",\n            "pip",\n            "install",\n            "-q",\n            "nvidia-cublas-cu12",\n            "nvidia-cudnn-cu12",\n        ]\n    )\n    roots = []\n    try:\n        roots.extend(site.getsitepackages())\n    except AttributeError:\n        pass\n    user_site = site.getusersitepackages()\n    if user_site:\n        roots.append(user_site)\n    lib_dirs = sorted(\n        {\n            os.path.dirname(path)\n            for root in roots\n            for path in glob.glob(f"{root}/nvidia/**/*.so*", recursive=True)\n        }\n    )\n    if not lib_dirs:\n        raise RuntimeError("Installed NVIDIA wheels, but libcublas was not found.")\n    current = os.environ.get("LD_LIBRARY_PATH", "")\n    os.environ["LD_LIBRARY_PATH"] = ":".join([*lib_dirs, current] if current else lib_dirs)\n\n\ndef gpu_settings(gpu_name: str) -> tuple[str, int]:\n    """Pick a compute type and batch size that fit a free GPU."""\n    name = gpu_name.lower()\n    if "p100" in name:\n        return "int8", 4\n    return "float16", 8\n\n\ndef default_output_path(source: Path) -> Path:\n    if Path("/kaggle/working").is_dir():\n        return Path("/kaggle/working") / f"{source.stem}.txt"\n    output_directory = Path("output")\n    if output_directory.exists() or not Path("/content").is_dir():\n        output_directory.mkdir(exist_ok=True)\n        return output_directory / f"{source.stem}.txt"\n    return Path("/content") / f"{source.stem}.txt"\n\n\ndef transcribe(\n    video_path: str | Path,\n    output_path: str | Path | None = None,\n    *,\n    model_size: str = "large-v3",\n    language: str | None = None,\n    offset_seconds: int = 0,\n    no_speech_threshold: float = 0.6,\n) -> Path:\n    import time\n\n    from faster_whisper import BatchedInferencePipeline, WhisperModel\n\n    source = Path(video_path)\n    if not source.exists():\n        raise FileNotFoundError(source)\n    destination = Path(output_path) if output_path else default_output_path(source)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n\n    if model_size == "large":\n        model_size = "large-v3"\n\n    gpu_name = require_gpu()\n    compute_type, batch_size = gpu_settings(gpu_name)\n    print(f"GPU: {gpu_name} ({compute_type}, batch {batch_size})", flush=True)\n    ensure_cuda_libs()\n\n    print(f"Converting to MP3: {source}", flush=True)\n    mp3_path = extract_mp3(source, offset_seconds)\n    try:\n        mp3_mb = mp3_path.stat().st_size / 1_048_576\n        print(f"MP3 ready: {mp3_mb:.1f} MB", flush=True)\n\n        started = time.perf_counter()\n        model = WhisperModel(model_size, device="cuda", compute_type=compute_type)\n        pipeline = BatchedInferencePipeline(model=model)\n        kept = _transcribe_batches(\n            pipeline,\n            mp3_path,\n            batch_size,\n            language,\n            no_speech_threshold,\n        )\n    finally:\n        mp3_path.unlink(missing_ok=True)\n\n    text = " ".join(kept).strip()\n    destination.write_text(text + "\\n", encoding="utf-8")\n    elapsed = time.perf_counter() - started\n    print(f"Transcript saved to: {destination} ({elapsed:.0f}s)", flush=True)\n    return destination\n\n\ndef _transcribe_batches(pipeline, mp3_path: Path, batch_size: int, language, no_speech_threshold: float) -> list[str]:\n    last_error: RuntimeError | None = None\n    sizes: list[int] = []\n    for size in (batch_size, 4, 1):\n        if size not in sizes:\n            sizes.append(size)\n    for size in sizes:\n        try:\n            segments, info = pipeline.transcribe(\n                str(mp3_path),\n                batch_size=size,\n                beam_size=1,\n                vad_filter=True,\n                condition_on_previous_text=False,\n                no_speech_threshold=no_speech_threshold,\n                language=language,\n            )\n            print(\n                f"Language: {info.language} ({info.language_probability:.2f})",\n                flush=True,\n            )\n            kept: list[str] = []\n            for index, segment in enumerate(segments):\n                if index % 25 == 0:\n                    print(f"segment {index} at {segment.start:.0f}s", flush=True)\n                if segment.no_speech_prob >= no_speech_threshold:\n                    continue\n                text = segment.text.strip()\n                if text.lower().rstrip(".,!? ") in HALLUCINATED_PHRASES:\n                    continue\n                kept.append(text)\n            print(f"Transcribed {info.duration:.0f}s of audio", flush=True)\n            return kept\n        except RuntimeError as exc:\n            if "memory" not in str(exc).lower():\n                raise\n            last_error = exc\n            try:\n                import torch\n\n                if torch.cuda.is_available():\n                    torch.cuda.empty_cache()\n            except ImportError:\n                pass\n            print(f"Batch size {size} ran out of GPU memory, retrying smaller.", flush=True)\n    assert last_error is not None\n    raise last_error\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="Transcribe a video on a CUDA GPU.")\n    parser.add_argument("video", type=Path, help="Video or audio file")\n    parser.add_argument("--output", type=Path, default=None, help="Transcript path")\n    parser.add_argument("--model", default="large-v3", help="faster-whisper model (default: large-v3)")\n    parser.add_argument("--language", default=None, help="Language code, e.g. en")\n    parser.add_argument("--offset", type=int, default=0, help="Seconds to skip at the start")\n    parser.add_argument("--no-speech-threshold", type=float, default=0.6)\n    args = parser.parse_args()\n    transcribe(\n        args.video,\n        args.output,\n        model_size=args.model,\n        language=args.language,\n        offset_seconds=args.offset,\n        no_speech_threshold=args.no_speech_threshold,\n    )\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
for name in ('audio_extract', 'gpu_transcribe'):
    sys.modules.pop(name, None)


In [ ]:
VIDEO = ""  # Example: "/kaggle/input/your-dataset/meeting.mkv"

from pathlib import Path

from gpu_transcribe import VIDEO_SUFFIXES, transcribe

if not VIDEO:
    found = [
        path
        for path in Path("/kaggle/input").rglob("*")
        if path.is_file() and path.suffix.lower() in VIDEO_SUFFIXES
    ]
    if len(found) != 1:
        listed = "\n".join(str(path) for path in found) or "(none)"
        raise SystemExit(f"Set VIDEO to one file. Found:\n{listed}")
    VIDEO = str(found[0])

transcript = transcribe(VIDEO)
print(transcript)
